<a href="https://colab.research.google.com/github/Terrence536/MSSP-6070/blob/main/Assignments/case_study_report_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# process the *data*

In [ ]:
from google.colab import drive
import pandas as pd

drive.mount("/content/drive")

file_path = "/content/drive/MyDrive/MSSP 6070/data/data_academic_performance.xlsx"

df = pd.read_excel(
    file_path,
    sheet_name="SABER11_SABERPRO"
)

print(df.shape)
display(df.head())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
(12411, 45)


,COD_S11,GENDER,EDU_FATHER,EDU_MOTHER,OCC_FATHER,OCC_MOTHER,STRATUM,SISBEN,PEOPLE_HOUSE,Unnamed: 9,...,CC_PRO,ENG_PRO,WC_PRO,FEP_PRO,G_SC,PERCENTILE,2ND_DECILE,QUARTILE,SEL,SEL_IHE
0,SB11201210000129,F,Incomplete Professional Education,Complete technique or technology,Technical or professional level employee,Home,Stratum 4,It is not classified by the SISBEN,Three,NaN,...,71,93,79,181,180,91,5,4,2,2
1,SB11201210000137,F,Complete Secundary,Complete professional education,Entrepreneur,Independent professional,Stratum 5,It is not classified by the SISBEN,Three,NaN,...,86,98,78,201,182,92,5,4,4,4
2,SB11201210005154,M,Not sure,Not sure,Independent,Home,Stratum 2,Level 2,Five,NaN,...,18,43,22,113,113,7,1,1,1,1
3,SB11201210007504,F,Not sure,Not sure,Other occupation,Independent,Stratum 2,It is not classified by the SISBEN,Three,NaN,...,76,80,48,137,157,67,4,3,2,2
4,SB11201210007548,M,Complete professional education,Complete professional education,Executive,Home,Stratum 4,It is not classified by the SISBEN,One,NaN,...,98,100,71,189,198,98,5,4,4,2


In [ ]:
key_columns = ["G_SC", "GENDER", "STRATUM", "EDU_FATHER", "EDU_MOTHER"]

display(df[key_columns].head())

print("Missing values in key columns:")
print(df[key_columns].isna().sum())

,G_SC,GENDER,STRATUM,EDU_FATHER,EDU_MOTHER
0,180,F,Stratum 4,Incomplete Professional Education,Complete technique or technology
1,182,F,Stratum 5,Complete Secundary,Complete professional education
2,113,M,Stratum 2,Not sure,Not sure
3,157,F,Stratum 2,Not sure,Not sure
4,198,M,Stratum 4,Complete professional education,Complete professional education


Missing values in key columns:
G_SC          0
GENDER        0
STRATUM       0
EDU_FATHER    0
EDU_MOTHER    0
dtype: int64


In [ ]:
for column in ["GENDER", "STRATUM", "EDU_FATHER", "EDU_MOTHER"]:
    print(f"\n{column}")
    print(df[column].value_counts(dropna=False))


GENDER
GENDER
M    7368
F    5043
Name: count, dtype: int64

STRATUM
STRATUM
Stratum 3    4045
Stratum 2    4029
Stratum 1    1709
Stratum 4    1578
Stratum 5     633
Stratum 6     403
0              14
Name: count, dtype: int64

EDU_FATHER
EDU_FATHER
Complete professional education          3016
Complete Secundary                       2843
Complete technique or technology         1194
Incomplete Secundary                     1091
Postgraduate education                   1085
Complete primary                          824
Incomplete primary                        735
Incomplete Professional Education         425
Not sure                                  407
0                                         391
Incomplete technical or technological     277
Ninguno                                   123
Name: count, dtype: int64

EDU_MOTHER
EDU_MOTHER
Complete Secundary                       3106
Complete professional education          3059
Complete technique or technology         1495
Incomple

In [ ]:
analysis_df = df.copy()

analysis_df["STRATUM"] = analysis_df["STRATUM"].replace(
    {0: pd.NA, "0": pd.NA}
)

for column in ["EDU_FATHER", "EDU_MOTHER"]:
    analysis_df[column] = analysis_df[column].replace(
        {0: pd.NA, "0": pd.NA, "Not sure": pd.NA}
    )

print("Rows in the original data:", len(df))
print("\nUnknown categories in the analysis copy:")
print(analysis_df[["STRATUM", "EDU_FATHER", "EDU_MOTHER"]].isna().sum())

Rows in the original data: 12411

Unknown categories in the analysis copy:
STRATUM        14
EDU_FATHER    798
EDU_MOTHER    567
dtype: int64


# parental education and outcome (global score)

In [ ]:
for column in ["EDU_FATHER", "EDU_MOTHER"]:
    print(f"\nGlobal score by {column}:")

    summary = (analysis_df.groupby(column)["G_SC"]
        .agg(count="count", mean="mean", median="median")
        .round(2)
        .sort_values("mean", ascending=False))

    display(summary)


Global score by EDU_FATHER:


,count,mean,median
EDU_FATHER,,,
Postgraduate education,1085,176.99,179.0
Complete professional education,3016,167.77,169.0
Incomplete Professional Education,425,166.92,168.0
Complete technique or technology,1194,164.03,164.0
Incomplete technical or technological,277,162.10,162.0
Complete Secundary,2843,158.79,159.0
Ninguno,123,156.63,159.0
Incomplete Secundary,1091,156.54,157.0
Incomplete primary,735,155.73,156.0



Global score by EDU_MOTHER:


,count,mean,median
EDU_MOTHER,,,
Postgraduate education,997,175.64,178.0
Complete professional education,3059,168.85,170.0
Incomplete Professional Education,502,166.69,168.0
Complete technique or technology,1495,163.24,163.0
Incomplete technical or technological,341,161.88,164.0
Complete Secundary,3106,158.47,159.0
Incomplete Secundary,1056,155.67,156.0
Complete primary,713,155.34,155.0
Incomplete primary,541,154.85,155.0


### using s11 scores

In [ ]:
s11_scores = ["MAT_S11", "CR_S11", "CC_S11", "BIO_S11", "ENG_S11"]

print("Missing values in S11 scores:")
print(analysis_df[s11_scores].isna().sum())

Missing values in S11 scores:
MAT_S11    0
CR_S11     0
CC_S11     0
BIO_S11    0
ENG_S11    0
dtype: int64


In [ ]:
s11_scores = ["MAT_S11", "CR_S11", "CC_S11", "BIO_S11", "ENG_S11"]

for parent in ["EDU_FATHER", "EDU_MOTHER"]:
    summary = (analysis_df.groupby(parent).agg(count=("MAT_S11", "count"),
            **{score: (score, "mean") for score in s11_scores}).round(2))

    print(f"High school scores by {parent}:")
    display(summary)

High school scores by EDU_FATHER:


,count,MAT_S11,CR_S11,CC_S11,BIO_S11,ENG_S11
EDU_FATHER,,,,,,
Complete Secundary,2843,62.28,59.43,59.30,62.07,58.04
Complete primary,824,60.66,57.98,58.55,60.79,54.89
Complete professional education,3016,67.01,62.49,62.37,66.21,66.83
Complete technique or technology,1194,64.88,61.29,61.27,64.37,62.20
Incomplete Professional Education,425,66.27,62.38,62.16,65.69,65.19
Incomplete Secundary,1091,61.27,58.71,58.66,61.27,56.54
Incomplete primary,735,61.25,58.52,58.78,61.33,55.95
Incomplete technical or technological,277,63.39,60.61,59.99,63.72,60.26
Ninguno,123,60.71,58.27,58.24,61.54,57.57


High school scores by EDU_MOTHER:


,count,MAT_S11,CR_S11,CC_S11,BIO_S11,ENG_S11
EDU_MOTHER,,,,,,
Complete Secundary,3106,62.19,59.24,59.32,62.01,57.70
Complete primary,713,60.50,57.77,58.40,61.15,55.03
Complete professional education,3059,67.44,62.78,62.58,66.61,67.69
Complete technique or technology,1495,64.56,61.27,61.20,64.04,62.27
Incomplete Professional Education,502,65.99,62.80,62.26,65.16,64.30
Incomplete Secundary,1056,60.95,58.34,58.52,61.02,56.05
Incomplete primary,541,60.82,58.23,58.26,61.24,54.95
Incomplete technical or technological,341,64.04,61.49,60.76,63.87,61.13
Ninguno,34,63.50,57.26,57.06,63.59,56.41


In [ ]:
s11_scores = ["MAT_S11", "CR_S11", "CC_S11", "BIO_S11", "ENG_S11"]

analysis_df["S11_FIVE_SUBJECT_MEAN"] = analysis_df[s11_scores].mean(axis=1)

display(analysis_df[["S11_FIVE_SUBJECT_MEAN"]].head())

,S11_FIVE_SUBJECT_MEAN
0,76.2
1,82.4
2,45.4
3,59.8
4,79.6


In [ ]:
  for parent in ["EDU_FATHER", "EDU_MOTHER"]:
    summary = (analysis_df.groupby(parent)["S11_FIVE_SUBJECT_MEAN"]
        .agg(count="count", mean="mean", median="median")
        .round(2))

    summary = summary.iloc[[8, 6, 1, 5, 0, 7, 3, 4, 2, 9]]

    print(f"Five-subject S11 mean by {parent} (approximate education order):")
    display(summary)



Five-subject S11 mean by EDU_FATHER (approximate education order):


,count,mean,median
EDU_FATHER,,,
Ninguno,123,59.27,60.6
Incomplete primary,735,59.17,58.4
Complete primary,824,58.58,57.8
Incomplete Secundary,1091,59.29,58.6
Complete Secundary,2843,60.22,59.8
Incomplete technical or technological,277,61.59,61.8
Complete technique or technology,1194,62.80,62.2
Incomplete Professional Education,425,64.34,63.8
Complete professional education,3016,64.98,64.8


Five-subject S11 mean by EDU_MOTHER (approximate education order):


,count,mean,median
EDU_MOTHER,,,
Ninguno,34,59.56,58.9
Incomplete primary,541,58.70,58.2
Complete primary,713,58.57,57.6
Incomplete Secundary,1056,58.98,58.6
Complete Secundary,3106,60.09,59.6
Incomplete technical or technological,341,62.26,62.4
Complete technique or technology,1495,62.67,62.2
Incomplete Professional Education,502,64.10,63.8
Complete professional education,3059,65.42,65.0


# stratum and outcome (global score)

In [ ]:
stratum_order = [f"Stratum {i}" for i in range(1, 7)]

stratum_summary = (
    analysis_df.groupby("STRATUM")["G_SC"]
    .agg(count="count", mean="mean", median="median")
    .reindex(stratum_order)
    .round(2)
)

display(stratum_summary)

,count,mean,median
STRATUM,,,
Stratum 1,1709,152.35,152.0
Stratum 2,4029,158.20,158.0
Stratum 3,4045,163.82,164.0
Stratum 4,1578,172.00,174.0
Stratum 5,633,177.39,179.0
Stratum 6,403,181.51,184.0


### using s11 scores

In [ ]:
stratum_s11_mean = (analysis_df.groupby("STRATUM")["S11_FIVE_SUBJECT_MEAN"].agg(count="count", mean="mean", median="median")
.reindex([f"Stratum {i}" for i in range(1, 7)]).round(2))

display(stratum_s11_mean)

,count,mean,median
STRATUM,,,
Stratum 1,1709,57.57,56.8
Stratum 2,4029,59.95,59.4
Stratum 3,4045,62.69,62.2
Stratum 4,1578,66.85,66.8
Stratum 5,633,70.11,70.4
Stratum 6,403,72.48,72.8


# gender performance in global assessment

In [ ]:
gender_summary = (analysis_df.groupby("GENDER")["G_SC"]
    .agg(count="count", mean="mean", median="median")
    .round(2))

display(gender_summary)

,count,mean,median
GENDER,,,
F,5043,161.28,162.0
M,7368,163.69,164.0


In [ ]:
pro_scores = ["QR_PRO", "CR_PRO", "CC_PRO", "ENG_PRO", "WC_PRO", "FEP_PRO"]

gender_by_subject = (analysis_df.groupby("GENDER")[pro_scores]
    .mean().round(2))

display(gender_by_subject.T)

GENDER,F,M
QR_PRO,71.89,81.20
CR_PRO,61.34,62.79
CC_PRO,58.57,59.61
ENG_PRO,66.53,68.16
WC_PRO,56.62,51.71
FEP_PRO,145.30,145.60


In [ ]:
display(analysis_df[pro_scores].describe().T[["count", "min", "max", "mean"]])

print("Number of zero scores by subject:")
print((analysis_df[pro_scores] == 0).sum())

,count,min,max,mean
QR_PRO,12411.0,1.0,100.0,77.417291
CR_PRO,12411.0,1.0,100.0,62.199339
CC_PRO,12411.0,1.0,100.0,59.186770
ENG_PRO,12411.0,1.0,100.0,67.498348
WC_PRO,12411.0,0.0,100.0,53.703408
FEP_PRO,12411.0,1.0,300.0,145.476593


Number of zero scores by subject:
QR_PRO       0
CR_PRO       0
CC_PRO       0
ENG_PRO      0
WC_PRO     309
FEP_PRO      0
dtype: int64


In [ ]:
wc_zero_by_gender = (
    analysis_df.assign(WC_is_zero=analysis_df["WC_PRO"].eq(0))
    .groupby("GENDER")["WC_is_zero"]
    .agg(count="sum", total="size", proportion="mean")
)

wc_zero_by_gender["proportion"] = (
    wc_zero_by_gender["proportion"] * 100
).round(2)

display(wc_zero_by_gender)

,count,total,proportion
GENDER,,,
F,86,5043,1.71
M,223,7368,3.03


### excluding zeros

In [ ]:
wc_nonzero_summary = (analysis_df.loc[analysis_df["WC_PRO"] > 0]
    .groupby("GENDER")["WC_PRO"]
    .agg(count="count", mean="mean", median="median")
    .round(2))

display(wc_nonzero_summary)

,count,mean,median
GENDER,,,
F,4957,57.60,61.0
M,7145,53.32,55.0


### compare s11 scores

In [ ]:
gender_s11 = (
    analysis_df.groupby("GENDER")[s11_scores].mean().round(2))

gender_s11["S11 five-subject mean"] = (
    analysis_df.groupby("GENDER")["S11_FIVE_SUBJECT_MEAN"]
    .mean().round(2))

display(gender_s11.T)

GENDER,F,M
MAT_S11,61.93,65.96
CR_S11,60.90,60.70
CC_S11,59.96,61.22
BIO_S11,62.27,65.10
ENG_S11,61.45,62.04
S11 five-subject mean,61.30,63.00
